# Robotic Arm Strength Analysis
**ANYarm / Robotarm 1.mai** — Static structural analysis of CFRP tube links.

Dimensions measured from Fusion 360 assembly. Material properties from Easy Composites CFRP datasheet (thesis Ch. 7).

- **Link 2** (upper arm, shoulder → elbow): OD=50 mm, L=370 mm
- **Link 1** (forearm, elbow → wrist): OD=50 mm, L=340 mm

Run all cells top-to-bottom with **Run All** (`Ctrl+F9`).

In [ ]:
%matplotlib inline

import math
import json
import datetime
from dataclasses import dataclass, field
from typing import List, Tuple

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

print("Imports OK")

---
## 1 · Link Geometry
Dimensions from Fusion 360 (Inspect → Measure + Properties panel).

| Dim | Method | Value |
|-----|--------|-------|
| OD | Inspect → Measure on cylindrical face | 50 mm |
| ID | Back-calc from Fusion volume V = π(Ro²−Ri²)·L | 47 mm |
| Length | Cylindrical face area A = 2π·Ro·L | 370 / 340 mm |
| Mass | Fusion Properties panel (CFRP ρ ≈ 1430 kg/m³) | 121 / 111 g |

In [ ]:
@dataclass
class Link:
    name: str
    length_m: float
    mass_kg: float
    section: str           # 'circular_tube' | 'circular_solid' | 'rectangular_tube' | 'rectangular_solid'
    outer_diameter_m: float = 0.0
    inner_diameter_m: float = 0.0
    width_m: float  = 0.0
    height_m: float = 0.0
    wall_m: float   = 0.0
    material: str   = "aluminium_6061"


LINKS: List[Link] = [
    Link(
        name="Link 2 - Upper arm (shoulder->elbow)",
        length_m=0.370,
        mass_kg=0.121,          # 120.926 g — Fusion Properties
        section="circular_tube",
        outer_diameter_m=0.050,
        inner_diameter_m=0.047, # wall = 1.5 mm
        material="cfrp_easy_composites",
    ),
    Link(
        name="Link 1 - Forearm (elbow->wrist)",
        length_m=0.340,
        mass_kg=0.111,          # 111.121 g — Fusion Properties
        section="circular_tube",
        outer_diameter_m=0.050,
        inner_diameter_m=0.047,
        material="cfrp_easy_composites",
    ),
]

print(f"Links defined: {[l.name for l in LINKS]}")

---
## 2 · Motor & Payload Masses
Fusion 360 has incorrect material density for the motors — values below are manufacturer specs.

In [ ]:
# --- Motor masses (manufacturer / user-specified) ---
M_AKH70_48 = 1.400   # kg — shoulder motor (sits at base, does NOT load the tubes)
M_AKH70_16 = 0.860   # kg — elbow motor
M_AK45_36  = 0.340   # kg — each wrist motor (x3)

# --- 3-D printed housings (Fusion Properties are accurate for plastic parts) ---
M_ELBOW_HOUSING = 0.200   # kg
M_WRIST_HOUSING = 0.226   # kg — thesis table 7.3

# Concentrated mass at the DISTAL end of each link (one entry per link)
JOINT_MASSES_KG: List[float] = [
    M_AKH70_16 + M_ELBOW_HOUSING,      # 1.060 kg at elbow (end of Link 2)
    3 * M_AK45_36 + M_WRIST_HOUSING,   # 1.246 kg at wrist (end of Link 1)
]

# Motor names per joint (for labels)
JOINT_MOTOR_LABELS = ["AKH70-16 (elbow)", "3x AK45-36 (wrist)"]

# Joint names (proximal side of each link)
JOINT_NAMES = ["Shoulder (J1)", "Elbow (J2)"]

# --- End-effector ---
PAYLOAD_MASS_KG  = 1.331   # gripper assembly — Fusion Properties
GRIPPED_MASS_KG  = 0.500   # worst-case object being held
PAYLOAD_OFFSET_M = 0.060   # CoM offset beyond last joint [m]

G = 9.81  # m/s^2

print(f"Elbow joint mass : {JOINT_MASSES_KG[0]:.3f} kg")
print(f"Wrist joint mass : {JOINT_MASSES_KG[1]:.3f} kg")
print(f"Total payload    : {PAYLOAD_MASS_KG + GRIPPED_MASS_KG:.3f} kg")

---
## 3 · Materials

In [ ]:
@dataclass
class Material:
    name: str
    E_GPa: float
    G_GPa: float
    yield_MPa: float
    ultimate_MPa: float
    density_kg_m3: float


MATERIALS = {
    # CFRP is brittle — no yield plateau, so yield_MPa = ultimate_MPa = failure limit
    "cfrp_easy_composites": Material(
        "CFRP - Easy Composites (thesis)",
        E_GPa=64.0, G_GPa=5.0,
        yield_MPa=650, ultimate_MPa=650,  # thesis eq. 7.3 / Easy Composites datasheet
        density_kg_m3=1430,
    ),
    "aluminium_6061": Material("Aluminium 6061-T6", E_GPa=68.9,  G_GPa=26.0, yield_MPa=276, ultimate_MPa=310, density_kg_m3=2700),
    "aluminium_7075": Material("Aluminium 7075-T6", E_GPa=71.7,  G_GPa=26.9, yield_MPa=503, ultimate_MPa=572, density_kg_m3=2810),
    "steel_1045":     Material("Steel 1045",         E_GPa=200.0, G_GPa=80.0, yield_MPa=530, ultimate_MPa=625, density_kg_m3=7850),
    "carbon_fiber":   Material("CFRP (generic)",     E_GPa=70.0,  G_GPa=5.0,  yield_MPa=600, ultimate_MPa=700, density_kg_m3=1550),
    "pla":            Material("PLA (3D printed)",   E_GPa=3.5,   G_GPa=1.3,  yield_MPa=50,  ultimate_MPa=65,  density_kg_m3=1240),
    "petg":           Material("PETG (3D printed)",  E_GPa=2.1,   G_GPa=0.8,  yield_MPa=42,  ultimate_MPa=53,  density_kg_m3=1270),
}

print("Materials defined:", list(MATERIALS.keys()))

---
## 4 · Load Cases & Analysis Parameters
Joint angles (degrees). `theta=0` → link horizontal; positive = upward.

In [ ]:
LOAD_CASES = [
    # (name, [theta_link2_deg, theta_link1_deg])
    ("Fully extended - worst case",  [0,    0  ]),
    ("Upper arm 45, forearm 0",      [45,   0  ]),
    ("Upper arm 90 (vertical)",      [90,   0  ]),
    ("Elbow bent 45",                [0,    45 ]),
    ("Elbow bent 90 (L-shape)",      [0,    90 ]),
    ("Mid-reach working pose",       [30,   45 ]),
    ("Overhead - arm up",            [90,  -45 ]),
    ("Folded compact",               [45,   90 ]),
]

DYNAMIC_FACTOR = 1.5   # multiplies gravity loads to account for acceleration
REQUIRED_SF    = 3.0   # minimum acceptable safety factor (thesis §7)

print(f"{len(LOAD_CASES)} load cases, dynamic factor={DYNAMIC_FACTOR}, required SF={REQUIRED_SF}")

---
## 5 · Core Engineering Functions

In [ ]:
def section_properties(link: Link) -> dict:
    """Return cross-section area A [m^2], I [m^4], Z [m^3], J [m^4], c [m]."""
    s = link.section
    if s == "circular_solid":
        d = link.outer_diameter_m
        A = math.pi * d**2 / 4
        I = math.pi * d**4 / 64
        J = math.pi * d**4 / 32
        c = d / 2

    elif s == "circular_tube":
        do, di = link.outer_diameter_m, link.inner_diameter_m
        if di >= do:
            raise ValueError(f"{link.name}: inner diameter must be less than outer")
        A = math.pi * (do**2 - di**2) / 4
        I = math.pi * (do**4 - di**4) / 64
        J = math.pi * (do**4 - di**4) / 32
        c = do / 2

    elif s == "rectangular_solid":
        w, h = link.width_m, link.height_m
        A = w * h
        I = w * h**3 / 12
        J = (w * h * (w**2 + h**2)) / 12
        c = h / 2

    elif s == "rectangular_tube":
        w, h, t = link.width_m, link.height_m, link.wall_m
        wi, hi = w - 2*t, h - 2*t
        A = w*h - wi*hi
        I = (w*h**3 - wi*hi**3) / 12
        J = 2 * t * (w - t)**2 * (h - t)**2 / (w + h - 2*t)
        c = h / 2

    else:
        raise ValueError(f"Unknown section type: {s}")

    Z = I / c
    return {"A": A, "I": I, "Z": Z, "J": J, "c": c}


def forward_kinematics(links: List[Link], thetas_deg: List[float]) -> List[np.ndarray]:
    """Joint positions [x, y] in vertical plane. theta=0 -> link along +X."""
    positions = [np.array([0.0, 0.0])]
    angle = 0.0
    for link, theta in zip(links, thetas_deg):
        angle += math.radians(theta)
        dx = link.length_m * math.cos(angle)
        dy = link.length_m * math.sin(angle)
        positions.append(positions[-1] + np.array([dx, dy]))
    return positions


print("Section properties and kinematics functions defined.")

In [ ]:
def analyse_load_case(
    links: List[Link],
    thetas_deg: List[float],
    joint_masses_kg: List[float],
    payload_kg: float,
    payload_offset_m: float,
    dynamic_factor: float,
) -> List[dict]:
    """
    Static analysis — works proximal to distal.
    Returns list of result dicts, one per link (index 0 = closest to base).
    """
    positions = forward_kinematics(links, thetas_deg)

    ee_pos = positions[-1]
    angle_total = sum(math.radians(t) for t in thetas_deg)
    payload_pos = ee_pos + payload_offset_m * np.array([math.cos(angle_total), math.sin(angle_total)])

    results = []
    for i, link in enumerate(links):
        mat   = MATERIALS[link.material]
        props = section_properties(link)

        joint_pos = positions[i]
        next_pos  = positions[i + 1]

        distal_links = links[i:]
        distal_mass  = sum(l.mass_kg for l in distal_links) + payload_kg
        distal_mass += sum(joint_masses_kg[j] for j in range(i, len(joint_masses_kg)))

        # Bending moment at proximal joint from gravity
        M_gravity = 0.0

        # Distributed tube masses (CoM at midpoint of each link)
        for j, dl in enumerate(distal_links):
            com_pos = (positions[i + j] + positions[i + j + 1]) / 2
            lever = com_pos[0] - joint_pos[0]
            M_gravity += dl.mass_kg * G * lever

        # Concentrated joint masses
        for j in range(i, len(joint_masses_kg)):
            lever = positions[j + 1][0] - joint_pos[0]
            M_gravity += joint_masses_kg[j] * G * lever

        # Payload moment
        M_gravity += payload_kg * G * (payload_pos[0] - joint_pos[0])
        M_gravity  = abs(M_gravity) * dynamic_factor

        # Link axis angle
        link_angle = sum(math.radians(thetas_deg[k]) for k in range(i + 1))

        F_axial = distal_mass * G * math.sin(link_angle) * dynamic_factor
        F_shear = distal_mass * G * math.cos(link_angle) * dynamic_factor

        # Stresses
        sigma_bending = M_gravity * props["c"] / props["I"]  if props["I"] > 0 else 0.0
        sigma_axial   = abs(F_axial) / props["A"]             if props["A"] > 0 else 0.0
        tau_shear     = 2.0 * abs(F_shear) / props["A"]       if props["A"] > 0 else 0.0

        sigma_total = sigma_bending + sigma_axial
        von_mises   = math.sqrt(sigma_total**2 + 3 * tau_shear**2)

        sf_yield    = (mat.yield_MPa   * 1e6) / von_mises if von_mises > 0 else float("inf")
        sf_ultimate = (mat.ultimate_MPa * 1e6) / von_mises if von_mises > 0 else float("inf")

        # Cantilever tip deflection estimate
        F_tip = distal_mass * G * dynamic_factor
        deflection_m = (F_tip * link.length_m**3) / (3 * mat.E_GPa * 1e9 * props["I"]) if props["I"] > 0 else 0.0

        # Per-component torque contributions at this joint
        torque_components = {}
        for j, dl in enumerate(distal_links):
            com_pos = (positions[i + j] + positions[i + j + 1]) / 2
            lever = com_pos[0] - joint_pos[0]
            torque_components[f"Link {i+j+1} tube"] = dl.mass_kg * G * lever * dynamic_factor
        for j in range(i, len(joint_masses_kg)):
            lever = positions[j + 1][0] - joint_pos[0]
            torque_components[JOINT_MOTOR_LABELS[j]] = joint_masses_kg[j] * G * lever * dynamic_factor
        lever_p = payload_pos[0] - joint_pos[0]
        torque_components["Payload (gripper+obj)"] = payload_kg * G * lever_p * dynamic_factor

        results.append({
            "link":             link,
            "mat":              mat,
            "props":            props,
            "M_bending_Nm":     M_gravity,
            "F_axial_N":        F_axial,
            "F_shear_N":        F_shear,
            "T_joint_Nm":       M_gravity,
            "torque_components": torque_components,
            "sigma_bending_Pa": sigma_bending,
            "sigma_axial_Pa":   sigma_axial,
            "tau_shear_Pa":     tau_shear,
            "von_mises_Pa":     von_mises,
            "sf_yield":         sf_yield,
            "sf_ultimate":      sf_ultimate,
            "deflection_m":     deflection_m,
            "joint_pos":        joint_pos,
            "distal_pos":       next_pos,
        })

    return results


print("analyse_load_case defined.")

---
## 6 · Run Analysis

In [ ]:
total_payload = PAYLOAD_MASS_KG + GRIPPED_MASS_KG

all_results = {}
for case_name, angles in LOAD_CASES:
    if len(angles) != len(LINKS):
        raise ValueError(f"Load case '{case_name}' has {len(angles)} angles but arm has {len(LINKS)} links.")
    all_results[case_name] = analyse_load_case(
        LINKS, angles, JOINT_MASSES_KG,
        total_payload, PAYLOAD_OFFSET_M, DYNAMIC_FACTOR
    )

print(f"Analysed {len(all_results)} load cases across {len(LINKS)} links.")

---
## 7 · Results Report

In [ ]:
def stress_bar(sf: float, required: float) -> str:
    filled = min(int(sf / required * 10), 20)
    return f"[{'#' * filled + '.' * (20 - filled)}] SF={sf:.2f}"


def print_report(all_results: dict, required_sf: float, save_path: str = None):
    lines = []

    def out(s=""):
        lines.append(s)
        print(s)

    out("=" * 72)
    out("  ROBOTIC ARM STRENGTH ANALYSIS REPORT")
    out(f"  Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
    out("=" * 72)
    out(f"  Required SF: {required_sf}  |  Dynamic factor: {DYNAMIC_FACTOR}")
    out(f"  Payload: {PAYLOAD_MASS_KG} kg gripper + {GRIPPED_MASS_KG} kg object")
    out()

    for case_name, results in all_results.items():
        out(f"\n{'-'*72}")
        out(f"  LOAD CASE: {case_name}")
        out(f"{'-'*72}")
        all_pass = True

        for r in results:
            link = r["link"]
            mat  = r["mat"]
            sf_y = r["sf_yield"]
            sf_u = r["sf_ultimate"]
            sf   = min(sf_y, sf_u)
            status = "PASS" if sf >= required_sf else "FAIL"
            if sf < required_sf:
                all_pass = False

            out(f"\n  {link.name}  [{mat.name}]")
            out(f"    Area: {r['props']['A']*1e6:.2f} mm2  |  I = {r['props']['I']*1e12:.3f} mm4")
            out(f"    Bending M  : {r['M_bending_Nm']:.3f} N·m")
            out(f"    Shear F    : {r['F_shear_N']:.2f} N  |  Axial F: {r['F_axial_N']:.2f} N")
            out(f"    Joint torque: {r['T_joint_Nm']:.3f} N·m")
            out(f"    sigma_bending: {r['sigma_bending_Pa']/1e6:.2f} MPa")
            out(f"    sigma_axial  : {r['sigma_axial_Pa']/1e6:.2f} MPa")
            out(f"    tau_shear    : {r['tau_shear_Pa']/1e6:.2f} MPa")
            out(f"    Von Mises    : {r['von_mises_Pa']/1e6:.2f} MPa  (limit: {mat.ultimate_MPa} MPa)")
            out(f"    SF (yield)   : {stress_bar(sf_y, required_sf)}")
            out(f"    SF (ult.)    : {stress_bar(sf_u, required_sf)}")
            out(f"    Deflection   : {r['deflection_m']*1000:.3f} mm")
            out(f"    Status       : [{status}]")

        out(f"\n  Overall: {'ALL PASS' if all_pass else 'FAILURES DETECTED'}")

    out("\n" + "=" * 72)
    out("  SUMMARY - minimum safety factor across all load cases")
    out("=" * 72)
    out(f"  {'Link':<30} {'Min SF yield':<15} {'Min SF ult.':<13} Status")
    out(f"  {'-'*30} {'-'*14} {'-'*12} {'-'*6}")

    link_names = [r["link"].name for r in next(iter(all_results.values()))]
    for i, name in enumerate(link_names):
        min_sf_y = min(all_results[c][i]["sf_yield"]    for c in all_results)
        min_sf_u = min(all_results[c][i]["sf_ultimate"] for c in all_results)
        status = "PASS" if min(min_sf_y, min_sf_u) >= required_sf else "FAIL"
        out(f"  {name:<30} {min_sf_y:<15.2f} {min_sf_u:<13.2f} {status}")

    if save_path:
        with open(save_path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))
        print(f"\nReport saved -> {save_path}")


print_report(all_results, REQUIRED_SF, save_path="robot_arm_report.txt")

---
## 8 · Structural Plots (stress, safety factors, deflection)

In [ ]:
cases    = list(all_results.keys())
n_links  = len(LINKS)
colors   = plt.cm.tab10(np.linspace(0, 1, len(cases)))

fig = plt.figure(figsize=(16, 10))
fig.suptitle("Robotic Arm Strength Analysis", fontsize=14, fontweight="bold")
gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# --- Arm poses ---
ax_pose = fig.add_subplot(gs[0, 0])
ax_pose.set_title("Arm Poses (load cases)")
ax_pose.set_aspect("equal")
for idx, (case_name, angles) in enumerate(LOAD_CASES):
    pos = forward_kinematics(LINKS, angles)
    xs = [p[0] for p in pos]
    ys = [p[1] for p in pos]
    ax_pose.plot(xs, ys, "o-", color=colors[idx], linewidth=2, markersize=5, label=case_name[:22])
ax_pose.axhline(0, color="gray", linewidth=0.5)
ax_pose.axvline(0, color="gray", linewidth=0.5)
ax_pose.set_xlabel("X [m]")
ax_pose.set_ylabel("Y [m]")
ax_pose.legend(fontsize=6, loc="upper right")

# --- Safety factors ---
ax_sf = fig.add_subplot(gs[0, 1:])
ax_sf.set_title("Safety Factor (yield) - all load cases")
x     = np.arange(n_links)
width = 0.8 / len(cases)
for idx, (case_name, results) in enumerate(all_results.items()):
    sfs = [r["sf_yield"] for r in results]
    ax_sf.bar(x + idx * width, sfs, width, label=case_name[:22], color=colors[idx], alpha=0.85)
ax_sf.axhline(REQUIRED_SF, color="red", linestyle="--", linewidth=1.5, label=f"Required SF={REQUIRED_SF}")
ax_sf.set_xticks(x + width * len(cases) / 2)
ax_sf.set_xticklabels([f"Link {i+1}" for i in range(n_links)], fontsize=9)
ax_sf.set_ylabel("Safety Factor")
ax_sf.legend(fontsize=7)

# --- Von Mises heatmap ---
ax_hm = fig.add_subplot(gs[1, 0])
ax_hm.set_title("Von Mises Stress [MPa]")
data = np.array([[r["von_mises_Pa"] / 1e6 for r in all_results[c]] for c in cases])
im = ax_hm.imshow(data, aspect="auto", cmap="hot_r")
ax_hm.set_xticks(range(n_links))
ax_hm.set_xticklabels([f"L{i+1}" for i in range(n_links)], fontsize=8)
ax_hm.set_yticks(range(len(cases)))
ax_hm.set_yticklabels([c[:22] for c in cases], fontsize=6)
plt.colorbar(im, ax=ax_hm)
for i in range(len(cases)):
    for j in range(n_links):
        ax_hm.text(j, i, f"{data[i,j]:.1f}", ha="center", va="center",
                   fontsize=7, color="white" if data[i,j] > data.max()*0.5 else "black")

# --- Bending moment ---
ax_bm = fig.add_subplot(gs[1, 1])
ax_bm.set_title("Bending Moment at proximal joint [N·m]")
for idx, (case_name, results) in enumerate(all_results.items()):
    bms = [r["M_bending_Nm"] for r in results]
    ax_bm.plot(range(1, n_links+1), bms, "o-", color=colors[idx], label=case_name[:20], linewidth=1.5)
ax_bm.set_xlabel("Link #")
ax_bm.set_ylabel("Moment [N·m]")
ax_bm.legend(fontsize=6)

# --- Deflection ---
ax_def = fig.add_subplot(gs[1, 2])
ax_def.set_title("Tip Deflection (cantilever est.) [mm]")
for idx, (case_name, results) in enumerate(all_results.items()):
    defs = [r["deflection_m"] * 1000 for r in results]
    ax_def.plot(range(1, n_links+1), defs, "s-", color=colors[idx], label=case_name[:20], linewidth=1.5)
ax_def.set_xlabel("Link #")
ax_def.set_ylabel("Deflection [mm]")
ax_def.legend(fontsize=6)

plt.tight_layout()
plt.savefig("robot_arm_strength.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 9 · Joint Torques
The torque each motor must provide to hold the arm statically. Includes dynamic factor.

- **Joint 1 (shoulder, AKH70-48)** — must resist the full arm weight
- **Joint 2 (elbow, AKH70-16)** — must resist forearm + wrist + gripper weight

In [ ]:
# --- Joint torque summary table ---
header = f"{'Load Case':<32}" + "".join(f"  {JOINT_NAMES[i]:>20}" for i in range(len(LINKS)))
print(header)
print("-" * len(header))

for case_name, results in all_results.items():
    torques = [r["T_joint_Nm"] for r in results]
    row = f"{case_name:<32}" + "".join(f"  {t:>18.2f} Nm" for t in torques)
    print(row)

print()
print("=" * len(header))
print("WORST CASE PER JOINT:")
for i, link in enumerate(LINKS):
    max_t  = max(all_results[c][i]["T_joint_Nm"] for c in all_results)
    worst  = max(all_results, key=lambda c: all_results[c][i]["T_joint_Nm"])
    min_t  = min(all_results[c][i]["T_joint_Nm"] for c in all_results)
    print(f"  {JOINT_NAMES[i]}: max={max_t:.2f} Nm  [{worst}]   min={min_t:.2f} Nm")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Required Joint Torques (static + dynamic factor {DYNAMIC_FACTOR}x)", fontsize=13, fontweight="bold")

cases_list = list(all_results.keys())
colors_t   = plt.cm.tab10(np.linspace(0, 1, len(cases_list)))
x          = np.arange(len(LINKS))
width      = 0.8 / len(cases_list)

# --- Grouped bar: torque per joint per load case ---
ax1 = axes[0]
for idx, (case_name, results) in enumerate(all_results.items()):
    torques = [r["T_joint_Nm"] for r in results]
    ax1.bar(x + idx * width, torques, width, label=case_name[:24], color=colors_t[idx], alpha=0.85)
ax1.set_xticks(x + width * len(cases_list) / 2)
ax1.set_xticklabels(JOINT_NAMES, fontsize=10)
ax1.set_ylabel("Required Torque [N·m]")
ax1.set_title("Torque by Joint and Load Case")
ax1.legend(fontsize=7, loc="upper right")
ax1.grid(axis="y", alpha=0.3)

# --- Line: torque sensitivity to pose ---
ax2 = axes[1]
for i in range(len(LINKS)):
    torques = [all_results[c][i]["T_joint_Nm"] for c in cases_list]
    ax2.plot(range(len(cases_list)), torques, "o-", label=f"{JOINT_NAMES[i]}\n({JOINT_MOTOR_LABELS[i]})",
             linewidth=2, markersize=7)
ax2.set_xticks(range(len(cases_list)))
ax2.set_xticklabels([c[:20] for c in cases_list], rotation=35, ha="right", fontsize=8)
ax2.set_ylabel("Required Torque [N·m]")
ax2.set_title("Torque Sensitivity to Arm Pose")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("joint_torques.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 10 · Bending Moment Diagrams
Bending moment plotted continuously along the arm length for every load case.
The moment at each section = sum of (gravity force × horizontal lever arm) of all distal loads.

In [ ]:
def moment_diagram_points(
    links: List[Link],
    thetas_deg: List[float],
    joint_masses_kg: List[float],
    payload_kg: float,
    payload_offset_m: float,
    dynamic_factor: float,
    n_points: int = 80,
) -> List[Tuple[np.ndarray, np.ndarray, str]]:
    """
    Compute bending moment at n_points along each link.
    Returns: list of (arc_positions_mm, M_values_Nm, link_name) per link.
    arc_positions are cumulative distance from base in mm.
    """
    positions = forward_kinematics(links, thetas_deg)
    angle_total = sum(math.radians(t) for t in thetas_deg)
    payload_pos = positions[-1] + payload_offset_m * np.array([math.cos(angle_total), math.sin(angle_total)])

    link_diagrams = []
    cumulative_mm = 0.0

    for i, link in enumerate(links):
        angle_i = sum(math.radians(thetas_deg[k]) for k in range(i + 1))
        unit_x  = math.cos(angle_i)  # x-component of link direction

        s_vals = np.linspace(0.0, link.length_m, n_points)
        M_vals = np.zeros(n_points)

        for k, s in enumerate(s_vals):
            cut_x = positions[i][0] + s * unit_x
            M = 0.0

            # Remaining portion of link i beyond the cut
            remaining_frac = (link.length_m - s) / link.length_m
            com_x = positions[i][0] + (s + link.length_m) / 2 * unit_x
            M += link.mass_kg * remaining_frac * G * (com_x - cut_x)

            # All links distal to link i
            for j in range(i + 1, len(links)):
                angle_j = sum(math.radians(thetas_deg[q]) for q in range(j + 1))
                com_j_x = (positions[j][0] + positions[j + 1][0]) / 2
                M += links[j].mass_kg * G * (com_j_x - cut_x)

            # Concentrated joint masses distal to link i
            for j in range(i, len(joint_masses_kg)):
                M += joint_masses_kg[j] * G * (positions[j + 1][0] - cut_x)

            # Payload
            M += payload_kg * G * (payload_pos[0] - cut_x)

            M_vals[k] = abs(M) * dynamic_factor

        arc_mm = cumulative_mm + s_vals * 1000
        link_diagrams.append((arc_mm, M_vals, link.name))
        cumulative_mm += link.length_m * 1000

    return link_diagrams


print("moment_diagram_points defined.")

In [ ]:
cases_list  = list(all_results.keys())
colors_m    = plt.cm.tab10(np.linspace(0, 1, len(cases_list)))
elbow_mm    = LINKS[0].length_m * 1000   # x-position of elbow joint
total_mm    = sum(l.length_m for l in LINKS) * 1000

# Worst case = load case with highest shoulder torque
worst_case_name  = max(all_results, key=lambda c: all_results[c][0]["T_joint_Nm"])
worst_case_angles = dict(LOAD_CASES)[worst_case_name]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Bending Moment Diagrams Along the Arm", fontsize=13, fontweight="bold")

# --- Left: all load cases ---
ax1 = axes[0]
for idx, (case_name, angles) in enumerate(LOAD_CASES):
    diagrams = moment_diagram_points(LINKS, angles, JOINT_MASSES_KG,
                                     total_payload, PAYLOAD_OFFSET_M, DYNAMIC_FACTOR)
    first = True
    for arc_mm, M_vals, _ in diagrams:
        ax1.plot(arc_mm, M_vals, color=colors_m[idx], linewidth=1.8,
                 label=case_name[:24] if first else "")
        ax1.fill_between(arc_mm, M_vals, alpha=0.07, color=colors_m[idx])
        first = False

ax1.axvline(elbow_mm, color="gray", linestyle="--", linewidth=1.2, label="Elbow joint")
ax1.set_xlabel("Distance from shoulder [mm]")
ax1.set_ylabel("Bending Moment [N·m]")
ax1.set_title("All Load Cases")
ax1.legend(fontsize=7, loc="upper right")
ax1.grid(alpha=0.3)
ax1.set_xlim(0, total_mm)

# --- Right: worst case with component breakdown shading ---
ax2 = axes[1]
diagrams = moment_diagram_points(LINKS, worst_case_angles, JOINT_MASSES_KG,
                                 total_payload, PAYLOAD_OFFSET_M, DYNAMIC_FACTOR)
link_colors = ["#2980b9", "#e74c3c"]
for (arc_mm, M_vals, link_name), col in zip(diagrams, link_colors):
    ax2.plot(arc_mm, M_vals, color=col, linewidth=2.5, label=link_name)
    ax2.fill_between(arc_mm, M_vals, alpha=0.18, color=col)

# Mark joint values
for i, link in enumerate(LINKS):
    joint_mm = sum(LINKS[k].length_m for k in range(i)) * 1000
    t_val = all_results[worst_case_name][i]["T_joint_Nm"]
    ax2.annotate(f"{JOINT_NAMES[i]}\n{t_val:.1f} N·m",
                 xy=(joint_mm, t_val),
                 xytext=(joint_mm + 15, t_val * 1.05),
                 fontsize=8, color="black",
                 arrowprops=dict(arrowstyle="->", color="gray", lw=1))

ax2.axvline(elbow_mm, color="gray", linestyle="--", linewidth=1.2, label="Elbow joint")
ax2.set_xlabel("Distance from shoulder [mm]")
ax2.set_ylabel("Bending Moment [N·m]")
ax2.set_title(f"Worst case: {worst_case_name}")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
ax2.set_xlim(0, total_mm)

plt.tight_layout()
plt.savefig("moment_diagrams.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 11 · Torque Contribution Breakdown
For the worst-case pose, shows how much each component (link tube, motor, gripper) contributes to the required torque at each joint.
Use this to see which masses dominate motor sizing.

In [ ]:
worst_results = all_results[worst_case_name]

fig, axes = plt.subplots(1, len(LINKS), figsize=(14, 6))
fig.suptitle(f"Torque Contribution by Component\n(Worst case: {worst_case_name})",
             fontsize=12, fontweight="bold")

component_colors = [
    "#3498db", "#2ecc71", "#e74c3c", "#f39c12", "#9b59b6", "#1abc9c",
]

for i, link in enumerate(LINKS):
    ax = axes[i] if len(LINKS) > 1 else axes
    comps = worst_results[i]["torque_components"]

    labels = list(comps.keys())
    values = [abs(v) for v in comps.values()]
    total  = sum(values)

    # Stacked horizontal bar
    left = 0.0
    bar_height = 0.4
    for j, (label, val) in enumerate(zip(labels, values)):
        pct = val / total * 100
        color = component_colors[j % len(component_colors)]
        ax.barh(0, val, left=left, height=bar_height, color=color,
                label=f"{label}\n{val:.2f} N·m ({pct:.1f}%)")
        if pct > 5:  # only label segments wide enough to read
            ax.text(left + val / 2, 0, f"{pct:.0f}%", ha="center", va="center",
                    fontsize=8, color="white", fontweight="bold")
        left += val

    ax.set_xlim(0, total * 1.02)
    ax.set_yticks([])
    ax.set_xlabel("Torque contribution [N·m]")
    ax.set_title(f"{JOINT_NAMES[i]}\nTotal required: {total:.2f} N·m", fontsize=10)
    ax.legend(loc="lower right", fontsize=7, framealpha=0.9)
    ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("torque_contributions.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary
print(f"\nWorst case: {worst_case_name}")
for i, link in enumerate(LINKS):
    comps = worst_results[i]["torque_components"]
    total = sum(abs(v) for v in comps.values())
    dominant = max(comps, key=lambda k: abs(comps[k]))
    print(f"  {JOINT_NAMES[i]}: {total:.2f} N·m  (dominant: {dominant} = {abs(comps[dominant]):.2f} N·m, {abs(comps[dominant])/total*100:.0f}%)")

---
## 12 · Diameter Sweep
Find the minimum outer diameter that achieves the required safety factor.

In [ ]:
def sweep_outer_diameter(
    link_index: int,
    diameter_range_mm: List[float],
    wall_fraction: float = 0.03,
) -> None:
    """Vary OD of one link, report minimum SF across all load cases."""
    link = LINKS[link_index]
    if link.section != "circular_tube":
        print(f"Sweep only for circular_tube; {link.name} is {link.section}")
        return

    original_od = link.outer_diameter_m
    original_id = link.inner_diameter_m
    min_sfs = []

    for d_mm in diameter_range_mm:
        d = d_mm / 1000
        link.outer_diameter_m = d
        link.inner_diameter_m = d * (1 - 2 * wall_fraction)
        case_sfs = []
        for _, angles in LOAD_CASES:
            res = analyse_load_case(LINKS, angles, JOINT_MASSES_KG,
                                    total_payload, PAYLOAD_OFFSET_M, DYNAMIC_FACTOR)
            case_sfs.append(min(res[link_index]["sf_yield"], res[link_index]["sf_ultimate"]))
        min_sfs.append(min(case_sfs))

    link.outer_diameter_m = original_od
    link.inner_diameter_m = original_id

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(diameter_range_mm, min_sfs, "b-o", linewidth=2, markersize=4)
    ax.axhline(REQUIRED_SF, color="red", linestyle="--", label=f"Required SF = {REQUIRED_SF}")
    ax.axvline(original_od * 1000, color="green", linestyle=":", label=f"Actual OD = {original_od*1000:.0f} mm")
    ax.set_xlabel("Outer Diameter [mm]")
    ax.set_ylabel("Minimum Safety Factor (all cases)")
    ax.set_title(f"Diameter Sweep - {link.name}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"sweep_link{link_index+1}.png", dpi=150)
    plt.show()

    for d_mm, sf in zip(diameter_range_mm, min_sfs):
        if sf >= REQUIRED_SF:
            print(f"  Minimum sufficient OD: {d_mm:.1f} mm  (SF = {sf:.2f})")
            break
    else:
        print(f"  No diameter in range achieves SF >= {REQUIRED_SF}.")


sweep_outer_diameter(
    link_index=0,
    diameter_range_mm=list(range(20, 80, 2)),
    wall_fraction=0.03,
)

---
## 13 · Export Results to JSON

In [ ]:
export = {}
for case_name, results in all_results.items():
    export[case_name] = []
    for r in results:
        export[case_name].append({
            "link":              r["link"].name,
            "material":          r["mat"].name,
            "M_bending_Nm":      r["M_bending_Nm"],
            "T_joint_Nm":        r["T_joint_Nm"],
            "torque_components": r["torque_components"],
            "F_shear_N":         r["F_shear_N"],
            "F_axial_N":         r["F_axial_N"],
            "sigma_bending_MPa": r["sigma_bending_Pa"] / 1e6,
            "sigma_axial_MPa":   r["sigma_axial_Pa"]   / 1e6,
            "tau_shear_MPa":     r["tau_shear_Pa"]      / 1e6,
            "von_mises_MPa":     r["von_mises_Pa"]      / 1e6,
            "sf_yield":          r["sf_yield"],
            "sf_ultimate":       r["sf_ultimate"],
            "deflection_mm":     r["deflection_m"] * 1000,
        })

with open("robot_arm_results.json", "w") as f:
    json.dump(export, f, indent=2)

print("Saved -> robot_arm_results.json")